# CLIP-Guided Test-Time Optimization with GigaTok

This notebook demonstrates CLIP-guided image editing and token interpretability using GigaTok's 1D VQ tokenizer with hierarchical ViT encoder/decoder.

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

In [ ]:
import sys
if IN_COLAB:
    !git clone -q https://github.com/sbeeredd04/sandbox.git
    !pip install -q --progress-bar off jaxtyping open_clip_torch omegaconf timm
    sys.path.insert(0, "sandbox/token-opt")
    sys.path.insert(0, "sandbox/GigaTok")
else:
    # For local development
    sys.path.insert(0, "/home/sbeeredd/sandbox/token-opt")
    sys.path.insert(0, "/home/sbeeredd/sandbox/GigaTok")

print("Python path:")
for p in sys.path[:5]:
    print(f"  - {p}")

In [ ]:
import os
# Set this environment for deterministic execution
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

In [ ]:
import torch
# Enable for deterministic algorithms
torch.use_deterministic_algorithms(True, warn_only=False)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

from pathlib import Path
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as v2
import torchvision.transforms.v2.functional as tvf
from torchvision.datasets import ImageNet
from einops import rearrange

In [ ]:
from tto.test_time_opt import (
    TestTimeOpt,
    TestTimeOptConfig,
    CLIPObjective,
)

In [ ]:
gpus = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = gpus
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_gpus = torch.cuda.device_count()
print(f"Using device: {device}, num_gpus: {num_gpus}")

## Utils

In [ ]:
def load_img(path, device=None):
    if IN_COLAB:
        path = Path("sandbox/token-opt/notebooks") / Path(path)
    img = (1. / 255.) * torch.from_numpy(
        np.array(Image.open(path)).astype(np.float32)
    ).permute(2, 0, 1)
    img = tvf.resize(img, 256)
    img = tvf.center_crop(img, 256)
    img = img.unsqueeze(0)
    if device is not None:
        img = img.to(device)
    return img

def display_image(*tensors):
    tensors = [255. * t.squeeze() for t in tensors]
    img = Image.fromarray(rearrange(
        tensors, "b c h w -> h (b w) c"
    ).to("cpu", dtype=torch.uint8).numpy())
    display(img)

def opt_callback(info):
    if info.i % 50 == 0:
        print(f"i = {info.i}")
        print("  CLIP score =", "\t".join(
            map(lambda l: f"{-l:.3f}", info.loss))
        )
        imgs = tto.decode(info.tokens).clamp(0., 1.)
        display_image(*imgs)

# Set up the objective function

In [ ]:
# Use CLIP similarity maximization objective
objective = CLIPObjective(num_augmentations=8, cfg_scale=1.2)

# Set prompt
objective.prompt = [
    "a photo of a tiger",
    "a photo of a husky",
    "a photo of a sparrow",
]

# Optionally set a negative prompt
# Note: also need to set cfg_scale > 1 in CLIPObjective if using this!
objective.neg_prompt = "bad, low-res, unnatural"

# Configure test time optimization with GigaTok

GigaTok is a 1D VQ tokenizer with hierarchical ViT encoder/decoder architecture:
- **BL256**: Base encoder, Large decoder, 256 tokens (smaller, faster)
- **XLXXL256**: XL encoder, XXL decoder, 256 tokens (3B params, best quality)

The tokenizer compresses 256×256 images into 256 discrete tokens using:
1. CNN encoder → spatial features
2. ViT 2D→1D encoder → 256 latent tokens
3. Vector quantization → discrete codes (16,384 codebook entries)
4. ViT 1D→2D decoder → spatial features
5. CNN decoder → reconstructed image

In [ ]:
tto_config = TestTimeOptConfig(
    # Use GigaTok tokenizer
    # Format: "gigatok:MODEL_CONFIG" or "gigatok:MODEL_CONFIG:CHECKPOINT_PATH"
    # Available configs: BL256, XLXXL256
    titok_checkpoint="gigatok:BL256",  # Use BL256 for faster inference
    
    # Optimize in continuous space (before quantization)
    optimize_post_quantization_tokens=True,
    
    # Optimization parameters
    num_iter=1000,
    ema_decay=0.98,
    lr=0.1,
    enable_amp=True,
    reg_weight=0.025,
    reg_type="seed",
)
tto = TestTimeOpt(tto_config, objective).to(device)

# Load seed images

In [ ]:
# Load seed image
img = torch.cat([
    load_img("ILSVRC2012_val_00008636.png", device),
    load_img("ILSVRC2012_val_00008636.png", device),
    load_img("ILSVRC2012_val_00010240.png", device),
], dim=0)

# Alternatively, initialize directly from given tokens (e.g. randomly
# sampled), but this is disabled when setting `seed_tokens = None`.
seed_tokens = None

# Run Optimization

In [ ]:
print("Seed")
display_image(*img)

# Run optimization
torch.manual_seed(0)
img_opt = tto(
    seed=img if seed_tokens is None else None,
    seed_tokens=seed_tokens,
    callback=opt_callback
)

# Token Interpretability: Token Swapping

Let's explore what each token represents by swapping tokens between two images.

GigaTok uses **256 1D tokens** to represent an image. Unlike 2D tokenizers, these tokens are computed via cross-attention and don't have direct spatial correspondence, but we can still analyze their influence.

In [ ]:
from tto.gigatok_wrapper import load_gigatok_model
import imageio
from pathlib import Path
from IPython.display import Image as IPImage

# Load GigaTok model
gigatok_model = load_gigatok_model('BL256')
gigatok_model.eval()
gigatok_model.to(device)

In [ ]:
def encode_tokens_gigatok(model, img, return_quantized=False):
    with torch.no_grad():
        # Encode: (b, 3, 256, 256) -> (b, codebook_embed_dim, 1, num_latent_tokens)
        z = model.encode(img)
        
        if return_quantized:
            # Quantize tokens
            z, _ = model.quantize(z)
        
        return z


def decode_tokens_gigatok(model, tokens, are_quantized=False):
    with torch.no_grad():
        # If not already quantized, quantize before decoding
        if not are_quantized:
            tokens, _ = model.quantize(tokens)
        
        # Decode: (b, codebook_embed_dim, 1, num_latent_tokens) -> (b, 3, 256, 256)
        img = model.decode(tokens)
        return img


def swap_token_range(base_tokens, source_tokens, start_idx, end_idx):
    swapped = base_tokens.clone()
    swapped[:, :, :, start_idx:end_idx] = source_tokens[:, :, :, start_idx:end_idx]
    return swapped

In [ ]:
def save_img_tensor(tensor, path):
    tensor = tensor.squeeze().clamp(0., 1.) * 255.
    img_array = tensor.permute(1, 2, 0).cpu().to(dtype=torch.uint8).numpy()
    img = Image.fromarray(img_array)
    img.save(path)
    return img_array


def concat_images(img1, img2):
    img1_np = img1.squeeze().clamp(0., 1.) * 255.
    img1_np = img1_np.permute(1, 2, 0).cpu().to(dtype=torch.uint8).numpy()

    img2_np = img2.squeeze().clamp(0., 1.) * 255.
    img2_np = img2_np.permute(1, 2, 0).cpu().to(dtype=torch.uint8).numpy()

    concat_images = np.concatenate([img1_np, img2_np], axis=1)
    return concat_images


# Create output directory
output_dir = Path("gigatok_token_swap_frames")
output_dir.mkdir(exist_ok=True)

# Set whether to quantize tokens immediately
use_quantized_tokens = True

# Load two images for token swapping
img1 = load_img("ILSVRC2012_val_00008636.png", device)
img2 = load_img("ILSVRC2012_val_00010240.png", device)

# Encode to tokens
tokens1_original = encode_tokens_gigatok(gigatok_model, img1, return_quantized=use_quantized_tokens)
tokens2_original = encode_tokens_gigatok(gigatok_model, img2, return_quantized=use_quantized_tokens)

# Get token dimensions
b, d, _, n = tokens1_original.shape
print(f"Token shape: {tokens1_original.shape}")
print(f"Number of tokens: {n}")
print(f"Codebook dimension: {d}")
print(f"{'='*60}\n")

# Collect frames for GIF
frames = []

# Swap tokens progressively (in chunks for efficiency)
chunk_size = 8  # Swap 8 tokens at a time for smoother animation
num_chunks = (n + chunk_size - 1) // chunk_size

print(f"Creating progressive token swap animation ({num_chunks} frames)...")

for chunk_idx in range(num_chunks + 1):
    end_idx = min(chunk_idx * chunk_size, n)
    print(f"Tokens swapped: {end_idx}/{n}...", end="\r")
    
    # Swap tokens [0:end_idx] between the two images
    pair1_swapped = swap_token_range(tokens1_original, tokens2_original, 0, end_idx)
    pair2_swapped = swap_token_range(tokens2_original, tokens1_original, 0, end_idx)
    
    # Decode both pairs
    img_pair1 = decode_tokens_gigatok(gigatok_model, pair1_swapped, are_quantized=use_quantized_tokens)
    img_pair2 = decode_tokens_gigatok(gigatok_model, pair2_swapped, are_quantized=use_quantized_tokens)
    
    # Create side-by-side images
    pair1_concat = concat_images(img1, img_pair1)
    pair2_concat = concat_images(img2, img_pair2)
    
    # Stack both pairs vertically
    combined_frame = np.concatenate([pair1_concat, pair2_concat], axis=0)
    
    frames.append(combined_frame)

print(f"\n{'='*60}")
print(f"Progressive token swapping complete!")
print(f"{'='*60}\n")

# Save as GIF
gif_path_combined = output_dir / "token_swap_progressive.gif"
imageio.mimsave(gif_path_combined, frames, duration=100, loop=0)

print(f"GIF saved at {gif_path_combined}")

# Display the combined GIF
print("\nProgressive token swap animation:")
display(IPImage(filename=str(gif_path_combined)))

## Individual Token Importance

Now let's see the effect of swapping individual tokens (or small groups) to understand token-level semantics.

In [ ]:
# Create output directory
output_dir = Path("gigatok_individual_token_swap")
output_dir.mkdir(exist_ok=True)

# Token groups to visualize (beginning, middle, end)
token_positions = [
    (0, 8, "First 8 tokens"),
    (64, 72, "Tokens 64-72"),
    (128, 136, "Middle tokens (128-136)"),
    (192, 200, "Tokens 192-200"),
    (248, 256, "Last 8 tokens"),
]

print("Testing individual token group swaps...\n")

frames = []

for start_idx, end_idx, description in token_positions:
    print(f"Processing: {description}")
    
    # Swap token range
    pair1_swapped = swap_token_range(tokens1_original, tokens2_original, start_idx, end_idx)
    pair2_swapped = swap_token_range(tokens2_original, tokens1_original, start_idx, end_idx)
    
    # Decode both pairs
    img_pair1 = decode_tokens_gigatok(gigatok_model, pair1_swapped, are_quantized=use_quantized_tokens)
    img_pair2 = decode_tokens_gigatok(gigatok_model, pair2_swapped, are_quantized=use_quantized_tokens)
    
    # Create side-by-side images
    pair1_concat = concat_images(img1, img_pair1)
    pair2_concat = concat_images(img2, img_pair2)
    
    # Stack both pairs vertically
    combined_frame = np.concatenate([pair1_concat, pair2_concat], axis=0)
    
    # Save individual frame
    frame_pil = Image.fromarray(combined_frame)
    frames.append(np.array(frame_pil))
    
    frame_path = output_dir / f"swap_tokens_{start_idx}_{end_idx}.png"
    frame_pil.save(frame_path)
    print(f"  Saved: {frame_path}")

print(f"\n{'='*60}")
print(f"Individual token swap complete!")
print(f"{'='*60}\n")

# Save as GIF
gif_path_combined = output_dir / "individual_token_swaps.gif"
imageio.mimsave(gif_path_combined, frames, duration=1500, loop=0)  # 1.5 sec per frame

print(f"GIF saved at {gif_path_combined}")

# Display the GIF
print("\nIndividual token group swap animation:")
display(IPImage(filename=str(gif_path_combined)))

In [ ]:

#list of classes and their converstions to classes
class_names = {
    "affenpinscher_n02110627" : "A photo of a husky dog", 
    "Afghan_hound_n02088094" : "A photo of a husky dog", 
    "African_crocodile_n01697457" : "A photo of a husky dog", 
    "African_grey_n01817953" : "A photo of an eagle", 
    "African_hunting_dog_n02116738" : "A photo of a husky dog", 
    "Airedale_n02096051" : "A photo of a husky dog", 
    "American_black_bear_n02133161" : "A photo of a husky dog", 
    "American_Staffordshire_terrier_n02093428" : "A photo of a husky dog", 
    "Appenzeller_n02107908" : "A photo of a husky dog",
    "Arctic_fox_n02120079": "A photo of a husky dog",
    "Australian_terrier_n02096294" : "A photo of a husky dog", 
    
}

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
import json
from PIL import Image
from pathlib import Path

# Dictionary to store selected classes and their prompts
class_names = {}

# Path to imagenet samples
samples_dir = Path("imagenet_samples")

# Get all class folders
class_folders = sorted([d for d in samples_dir.iterdir() if d.is_dir()])

print(f"Found {len(class_folders)} classes in imagenet_samples/")
print("=" * 80)
print("\nInteractive Class Selection")
print("=" * 80)
print("Instructions:")
print("  - First, enter the prompt that will be used for ALL selected classes")
print("  - Then view sample images from each class")
print("  - Input 1 to SELECT the class (will use the prompt you entered)")
print("  - Input 0 to SKIP the class")
print("  - Input 'q' to QUIT and save results")
print("=" * 80)

# Get the prompt ONCE at the beginning
custom_prompt = input("\nEnter the prompt for ALL selected classes (e.g., 'A photo of a husky dog'): ").strip()
if not custom_prompt:
    print("Error: Prompt cannot be empty!")
    raise ValueError("Prompt is required")

print(f"\nUsing prompt: '{custom_prompt}'")
print("=" * 80)

def display_class_samples(class_folder, num_samples=5):
    """Display sample images from a class folder"""
    image_files = sorted(list(class_folder.glob("*.JPEG")))[:num_samples]
    
    if not image_files:
        print(f"No images found in {class_folder.name}")
        return
    
    # Create a figure with subplots
    fig, axes = plt.subplots(1, len(image_files), figsize=(15, 3))
    if len(image_files) == 1:
        axes = [axes]
    
    for ax, img_path in zip(axes, image_files):
        img = Image.open(img_path)
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(img_path.name, fontsize=8)
    
    plt.suptitle(f"Class: {class_folder.name}", fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Load existing class_names if available
class_names_file = Path("class_names_mapping.json")
if class_names_file.exists():
    with open(class_names_file, 'r') as f:
        class_names = json.load(f)
    print(f"\nLoaded {len(class_names)} existing class mappings from {class_names_file}")
    print("=" * 80)

# Interactive loop
for idx, class_folder in enumerate(class_folders):
    folder_name = class_folder.name
    
    # Skip if already processed
    if folder_name in class_names:
        continue
    
    print(f"\n[{idx + 1}/{len(class_folders)}] Processing: {folder_name}")
    print("-" * 80)
    
    # Display sample images
    display_class_samples(class_folder)
    
    # Get user input
    while True:
        choice = input(f"\nSelect this class? (1=YES, 0=NO, q=QUIT): ").strip().lower()
        
        if choice == 'q':
            print("\nQuitting and saving results...")
            break
        elif choice == '1':
            # Use the global custom_prompt
            class_names[folder_name] = custom_prompt
            print(f"✓ Added: {folder_name} -> '{custom_prompt}'")
            break
        elif choice == '0':
            print(f"✗ Skipped: {folder_name}")
            break
        else:
            print("Invalid input. Please enter 1, 0, or q.")
    
    if choice == 'q':
        break
    
    # Save progress periodically (every 10 classes)
    if (idx + 1) % 10 == 0:
        with open(class_names_file, 'w') as f:
            json.dump(class_names, f, indent=2)
        print(f"\n[Progress saved: {len(class_names)} classes selected so far]")

# Final save
with open(class_names_file, 'w') as f:
    json.dump(class_names, f, indent=2)

print("\n" + "=" * 80)
print(f"COMPLETE! Selected {len(class_names)} classes for large-scale training")
print(f"Saved to: {class_names_file.absolute()}")
print("=" * 80)

# Display summary
print("\nSummary of selected classes:")
for i, (folder, prompt) in enumerate(list(class_names.items())[:10]):
    print(f"  {i+1}. {folder}: '{prompt}'")
if len(class_names) > 10:
    print(f"  ... and {len(class_names) - 10} more")

# Load Selected Classes and Prepare for Large-Scale Training

Now we'll load the selected classes from the JSON file and prepare them for batch CLIP-guided optimization.

In [ ]:
import json
from pathlib import Path

# Load the class mappings
class_names_file = Path("class_names_mapping.json")

if class_names_file.exists():
    with open(class_names_file, 'r') as f:
        class_names = json.load(f)
    print(f"Loaded {len(class_names)} class mappings")
    print("=" * 80)
    
    # Display statistics
    print("\nClass Selection Summary:")
    print(f"  Total classes selected: {len(class_names)}")
    print(f"  Total images available: {len(class_names) * 5} (5 per class)")
    print("=" * 80)
    
    # Show first 10 mappings
    print("\nFirst 10 class mappings:")
    for i, (folder, prompt) in enumerate(list(class_names.items())[:10]):
        print(f"  {i+1}. {folder}")
        print(f"     → '{prompt}'")
    
    if len(class_names) > 10:
        print(f"\n  ... and {len(class_names) - 10} more classes")
    
    print("\n" + "=" * 80)
    
    # Create a list of (image_path, prompt) pairs for training
    training_data = []
    samples_dir = Path("imagenet_samples")
    
    for folder_name, prompt in class_names.items():
        class_folder = samples_dir / folder_name
        if class_folder.exists():
            # Get all images from this class
            image_files = sorted(list(class_folder.glob("*.JPEG")))
            for img_path in image_files:
                training_data.append((str(img_path), prompt))
    
    print(f"\nPrepared {len(training_data)} image-prompt pairs for training")
    print("=" * 80)
    
    # Show some examples
    print("\nExample training pairs:")
    for i in range(min(5, len(training_data))):
        img_path, prompt = training_data[i]
        print(f"  {i+1}. {Path(img_path).name}")
        print(f"     → '{prompt}'")
    
else:
    print(f"No class mappings found at {class_names_file}")
    print("Please run the interactive class selection cell first!")
    class_names = {}
    training_data = []